# 03 · Carbon-aware scheduling vs a carbon-blind baseline

**Goal.** For each load type, compare a **carbon-blind** baseline against the
**carbon-minimising** schedule and report the carbon saved.

**Cost model.** Running a job at start hour $s$ for its `duration` hours costs

$$ \mathrm{cost}(s) \;=\; \text{power\_kw} \times \sum_{h=0}^{\text{dur}-1}\mathrm{CI}(s+h) $$

(kWh-weighted gCO2). The scheduler picks the cheapest feasible $s$ in each job's window.

- `fifo_baseline` — carbon-blind control: run every job at its `earliest_start`.
- `schedule` — greedy carbon-minimiser. **With no capacity cap the jobs never interact, so
  each takes its own cheapest hour: this is the exact per-job optimum**, identical to the
  MILP `schedule_optimal`.

Savings $= (\text{fifo} - \text{optimal}) / \text{fifo}$.

In [1]:
# --- standard setup for every notebook in this paper ------------------------
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, "..")                 # find cals (in ../)
from dotenv import load_dotenv; load_dotenv("../.env")  # loads EIA_API_KEY if present
from cals import *          # FACTORS, carbon_intensity, Job, schedule, fifo_baseline, ...
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import nb_utils as U             # guarded data loaders + figure helper (see notebooks/nb_utils.py)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

In [2]:
ci, source = U.get_carbon_intensity()
print("carbon source:", source)
hvac_jobs, run_hours, _ = U.load_hvac_jobs(flex_hours=6)
ev_jobs, ev_source = U.get_ev_jobs()
print("EV source:", ev_source)
ai_raw = U.load_pai_task_table(U.DATA / "ai" / "pai_task_table.csv")
ai_parsed = U.parse_alibaba_trace(ai_raw, origin=U.DEMO_ORIGIN)
ai_groups = aggregate_alibaba_tasks(ai_parsed)

carbon source: EIA ISO-NE (LIVE API)


EV: real ACN-Data unavailable (ModuleNotFoundError); using a labelled DEMO EV set. Install acnportal + set ACN_API_TOKEN for real sessions.


AI: real Alibaba GPU v2020 trace not found in data/ai/; using a labelled SYNTHETIC Alibaba-shaped trace.


## Per-load savings

Uncapacitated `schedule` == optimal, so the row below is the best any scheduler could do on
this carbon curve. (We also confirm greedy == the MILP `schedule_optimal` for HVAC.)

In [3]:
hvac_baseline = U.do_nothing_gco2(hvac_jobs, run_hours, ci)
hvac_opt = schedule(hvac_jobs, ci, baseline_hours=run_hours)
ev_fifo = fifo_baseline(ev_jobs, ci)
ev_opt = schedule(ev_jobs, ci)                     # uncapacitated == optimal
ai_costs = uncapped_alibaba_costs(ai_groups, ci, gpu_power_kw=0.4, flex_hours=6)
rows = [
    {"load": "HVAC", "jobs": len(hvac_jobs),
     "median slack h": round(float(np.median([slack_h(job) for job in hvac_jobs])), 2),
     "baseline gCO2": round(hvac_baseline),
     "optimal gCO2": round(hvac_opt["total_gco2"]),
     "savings %": round(U.savings_pct(hvac_baseline, hvac_opt["total_gco2"]), 2)},
    {"load": "EV", "jobs": len(ev_jobs),
     "median slack h": round(float(np.median([slack_h(job) for job in ev_jobs])), 2),
     "baseline gCO2": round(ev_fifo["total_gco2"]),
     "optimal gCO2": round(ev_opt["total_gco2"]),
     "savings %": round(U.savings_pct(ev_fifo["total_gco2"], ev_opt["total_gco2"]), 2)},
    {"load": "AI", "jobs": ai_costs["comparable_tasks"], "median slack h": 6.0,
     "baseline gCO2": round(ai_costs["baseline_gco2"]),
     "optimal gCO2": round(ai_costs["scheduled_gco2"]),
     "savings %": round(U.savings_pct(ai_costs["baseline_gco2"], ai_costs["scheduled_gco2"]), 2)},
]
table = pd.DataFrame(rows).set_index("load")
display(table)

# sanity check: greedy (uncapacitated) really is the exact optimum
g = schedule(hvac_jobs, ci)["total_gco2"]; m = schedule_optimal(hvac_jobs, ci)["total_gco2"]
print("HVAC greedy vs MILP optimal gCO2: %.1f vs %.1f  (equal => %s)" % (g, m, abs(g - m) < 1e-6))

,jobs,fifo gCO2,optimal gCO2,savings %
load,,,,
HVAC,8363,1219614,1082642,11.23
EV,200,1019546,977179,4.16
AI,299,415465,386482,6.98


HVAC greedy vs MILP optimal gCO2: 1082641.7 vs 1082641.7  (equal => True)


In [ ]:
# CURATED FIGURE (restores figures/03_savings_by_load.png as committed in 98f159a).
# Carries the job counts and the arm being reported; the bare default carried neither.
order = ["HVAC", "EV", "AI"]
pretty = {"HVAC": "HVAC\n(heat pump)", "EV": "EV\n(charging)", "AI": "Batch\n(AI compute)"}
vals = [float(table.loc[k, "savings %"]) for k in order]
ns = [int(table.loc[k, "jobs"]) for k in order]

fig, ax = plt.subplots(figsize=(6.5, 3.7))
ax.bar(range(3), vals, width=0.6, color=["#2a9d8f", "#e9c46a", "#e76f51"])
for i, v in enumerate(vals):
    ax.text(i, v + max(vals) * 0.02, f"{v:.2f}%", ha="center", fontweight="bold", fontsize=11)
ax.set_xticks(range(3))
ax.set_xticklabels([f"{pretty[k]}\nn={n:,}" for k, n in zip(order, ns)])
ax.set_ylim(0, max(vals) * 1.22)
ax.set_ylabel("carbon saved (%)")
# Every arm here is uncapped, and uncapped greedy is provably the exact optimum
# (verified in the cell above), so this is the ceiling, not a heuristic's achievement.
fig.tight_layout(); U.savefig(fig, "03_savings_by_load.png"); plt.show()
print("plotted: " + " | ".join(f"{k} {v:.2f}% (n={n:,})" for k, v, n in zip(order, vals, ns)))

**Reading it.** Savings track two things: how much **slack** a load has (notebook 02) and
how much the CI curve **swings** inside that slack (notebook 01). HVAC's many small, flexible
hours across a whole year give it steady savings; EV/AI depend on their observed workload windows here.
Notebook 04 sweeps the knobs behind these numbers.